# Librerias


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import rdata
from IPython.display import display


# URLS

Ejecutar con el directorio de trabajo en `entregas/`.


In [2]:
carpeta_entregas = Path.cwd()
data_folder = carpeta_entregas / "dataset"
carpeta_parametros = carpeta_entregas / "parametros"
carpeta_salida = carpeta_entregas / "entregas"
archivo_logging = carpeta_entregas / "logging_python.csv"


# Parametrizacion de la prueba


In [3]:
# dataset_file = "competencia_01_ternaria_lags.csv"
dataset_file = "competencia_01_ternaria_lf_k6_perdida4.csv"

train_final = [202103, 202104]
future = [202108]  # mes a predecir
SEEDS = [230047, 230059, 230063, 230077, 230081]

id_rds = 44  # archivo parametros/<id_rds>.rds
envios = 10500  # cantidad de clientes con Predicted = 1


In [4]:
columnas_excluir = [
    "ternaria",
    "ternaria_lag1",
    "clase01",
    "fold",
    "azar",
    "training",
    "cprestamos_personales",
    "mprestamos_personales",
    "cprestamos_personales_lag1",
    "mprestamos_personales_lag1",
    "cprestamos_personales_delta1",
    "mprestamos_personales_delta1",
    "cprestamos_personales_lag_pond",
    "mprestamos_personales_lag_pond",
    "cprestamos_personales_delta_lag_pond",
    "mprestamos_personales_delta_lag_pond"
]

if pd.isna([*future, id_rds, envios]).any():
    raise ValueError("Completar future, id_rds y envios en esta celda.")


# LEEMOS DATASET


In [5]:
dataset_url = data_folder / dataset_file
dataset = pd.read_csv(dataset_url, low_memory=False)
dataset["ternaria"] = dataset["ternaria"].fillna("")
dataset = dataset.sort_values(
    ["foto_mes", "ternaria", "numero_de_cliente"],
    kind="stable", na_position="first",
).reset_index(drop=True)

display(dataset.groupby(["foto_mes", "ternaria"], dropna=False).size().reset_index(name="N"))


,foto_mes,ternaria,N
0,202103,BAJA+1,1019
1,202103,BAJA+2,960
2,202103,continua,160921
3,202104,BAJA+1,964
4,202104,BAJA+2,1139
5,202104,continua,161181
6,202105,BAJA+1,1143
7,202105,BAJA+2,870
8,202105,continua,161755
9,202106,BAJA+1,874


# Parametros

El `.rds` debe estar copiado en `parametros/`. Se usan los parametros guardados, sin volver a ajustar `min_data_in_leaf`.


In [6]:

# rdata lee también las listas anidadas del RDS, sin requerir R.
# Convierte los vectores escalares de R en escalares de Python.
def convertir_r(valor):
    if isinstance(valor, dict):
        return {k: convertir_r(v) for k, v in valor.items()}
    if isinstance(valor, np.ndarray):
        if valor.size == 1:
            return convertir_r(valor.item())
        return [convertir_r(v) for v in valor.tolist()]
    if isinstance(valor, (list, tuple)):
        return [convertir_r(v) for v in valor]
    if isinstance(valor, np.generic):
        return convertir_r(valor.item())
    # R serializa muchos parámetros enteros como double (31.0).
    # LightGBM Python exige int para esos valores; se conserva su valor.
    if isinstance(valor, float) and np.isfinite(valor) and valor.is_integer():
        return int(valor)
    return valor

parametros = convertir_r(rdata.read_rds(carpeta_parametros / f"{id_rds}.rds"))


# Dataset train y prediccion


In [7]:
dataset["clase01"] = dataset["ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int32)

campos_buenos = [c for c in dataset.columns if c not in columnas_excluir]

dataset_train = dataset.loc[dataset["foto_mes"].isin(train_final)].copy()
dfuture = dataset.loc[dataset["foto_mes"].isin(future)].copy()

if (len(future) != 1 or dataset_train.empty or dfuture.empty
        or envios < 1 or envios > len(dfuture) or envios != int(envios)):
    raise ValueError("Revisar meses y envios: se requiere un mes a predecir y filas en ambos datasets.")

display(dataset_train.groupby(["foto_mes", "ternaria"], dropna=False).size().reset_index(name="N"))

# Equivalente a data.matrix: variables numéricas sin tratar como categóricas.
# Si existen columnas de texto, R las convierte a códigos de factor desde 1.
# Los niveles se calculan por separado para train y future, igual que en R.
def data_matrix(df):
    matriz = df.copy()
    for nombre in matriz.columns:
        serie = matriz[nombre]
        if isinstance(serie.dtype, pd.CategoricalDtype):
            codigos = serie.cat.codes
            matriz[nombre] = (codigos + 1).where(codigos >= 0, np.nan)
        elif pd.api.types.is_object_dtype(serie.dtype) or pd.api.types.is_string_dtype(serie.dtype):
            niveles = sorted(serie.dropna().unique())
            codigos = pd.Categorical(serie, categories=niveles).codes
            matriz[nombre] = np.where(codigos >= 0, codigos + 1, np.nan)
    return matriz.to_numpy(dtype=np.float64)

X_train = data_matrix(dataset_train[campos_buenos])
X_future = data_matrix(dfuture[campos_buenos])


/tmp/ipykernel_304367/2113917024.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset["clase01"] = dataset["ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(np.int32)


,foto_mes,ternaria,N
0,202103,BAJA+1,1019
1,202103,BAJA+2,960
2,202103,continua,160921
3,202104,BAJA+1,964
4,202104,BAJA+2,1139
5,202104,continua,161181


# Entreno y genero salidas


In [8]:
if archivo_logging.exists():
    historico = pd.read_csv(archivo_logging)
    id_prueba = int(historico["id_prueba"].max()) + 1
else:
    id_prueba = 1

carpeta_salida.mkdir(parents=True, exist_ok=True)
archivos_salida = []

for seed in SEEDS:
    # El .rds puede contener parametros individuales o una lista por semilla.
    if "num_iterations" in parametros:
        params_semilla = parametros.copy()
    else:
        params_semilla = parametros.get(str(seed))
        if params_semilla is None:
            raise ValueError(f"El .rds no contiene parametros para la semilla {seed}")
        params_semilla = params_semilla.copy()
    params_semilla["seed"] = seed

    num_boost_round = int(params_semilla.pop("num_iterations"))

    dtrain_final = lgb.Dataset(
        data=X_train,
        label=dataset_train["clase01"].to_numpy(),
        feature_name=campos_buenos,
        categorical_feature=[],
    )

    modelo_final = lgb.train(
        params=params_semilla,
        train_set=dtrain_final,
        num_boost_round=num_boost_round,
    )

    prediccion = modelo_final.predict(X_future)

    tb_prediccion = dfuture[["numero_de_cliente"]].copy()
    tb_prediccion["prob"] = prediccion
    tb_prediccion = tb_prediccion.sort_values(
        "prob", ascending=False, kind="stable"
    ).reset_index(drop=True)
    tb_prediccion["Predicted"] = 0
    tb_prediccion.loc[:envios - 1, "Predicted"] = 1

    archivos_salida.append(f"p001_{seed}_{id_prueba}.csv")
    tb_prediccion.loc[
        tb_prediccion["Predicted"].eq(1), ["numero_de_cliente"]
    ].to_csv(
        carpeta_salida / archivos_salida[-1],
        sep=",",
        header=False,
        index=False,
    )

    print(f"Semilla={seed} Archivo={archivos_salida[-1]}")


Semilla=230047 Archivo=p001_230047_6.csv
Semilla=230059 Archivo=p001_230059_6.csv
Semilla=230063 Archivo=p001_230063_6.csv
Semilla=230077 Archivo=p001_230077_6.csv
Semilla=230081 Archivo=p001_230081_6.csv


# Documentacion de la prueba

Una fila por ejecucion, con todas las semillas usadas. Completar la nota y los resultados de Kaggle como texto. Cuando lleguen los resultados, ejecutar las dos celdas de esta seccion: actualizan la misma fila y conservan las pruebas anteriores.


In [9]:
# cortamos aca
pepe

NameError: name 'pepe' is not defined

In [10]:
nota = "e4_p pasando de 11.5k envios a 10.5k envios, python"
resultado_competencia_mean = "95.4525"  # score de Kaggle o referencia a un problema con la entrega
resultado_competencia_std = "1.8443"
ref_submit = "e6_p"


In [11]:
registro_prueba = pd.DataFrame([{
    "id_prueba": id_prueba,
    "ref_submit": ref_submit,
    "notebook": "p001.ipynb",
    "dataset": dataset_file,
    "seeds": ",".join(map(str, SEEDS)),
    "cantidad_registros_totales": len(dataset),
    "cantidad_registros_entrenamiento": len(dataset_train),
    "cantidad_registros_prediccion": len(dfuture),
    "meses_entrenamiento": ",".join(map(str, train_final)),
    "meses_prediccion": ",".join(map(str, future)),
    "columnas_excluidas": ",".join(columnas_excluir),
    "bajas":"1+2",                                       #baja+1 y baja+2 o solo baja+2
    "id_rds": id_rds,
    "envios": envios,
    "archivos": ",".join(archivos_salida),
    "nota": nota,
    "resultado_competencia_mean": resultado_competencia_mean,
    "resultado_competencia_std": resultado_competencia_std
}])

if archivo_logging.exists():
    historico = pd.read_csv(
        archivo_logging,
        dtype={
            "ref_submit": str,
            "seeds": str,
            "meses_entrenamiento": str,
            "meses_prediccion": str,
            "nota": str,
            "resultado_competencia_mean": str,
            "resultado_competencia_std": str,
        },
    )
    fila = historico["id_prueba"].eq(id_prueba)

    if fila.any():
        historico.loc[fila, registro_prueba.columns] = registro_prueba.to_numpy()
        historico.to_csv(archivo_logging, index=False)
    else:
        registro_prueba.to_csv(archivo_logging, mode="a", header=False, index=False)
else:
    registro_prueba.to_csv(archivo_logging, index=False)

# display(registro_prueba)


In [12]:
display(pd.read_csv(archivo_logging).tail(10))

,id_prueba,ref_submit,notebook,dataset,seeds,cantidad_registros_totales,cantidad_registros_entrenamiento,cantidad_registros_prediccion,meses_entrenamiento,meses_prediccion,columnas_excluidas,bajas,id_rds,envios,archivos,nota,resultado_competencia_mean,resultado_competencia_std
0,1,1st_test_python,p001.ipynb,competencia_01_ternaria_lags.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,train...",1+2,44,11500,"p001_230047_1.csv,p001_230059_1.csv,p001_23006...",prueba equivalente a 1st_test pero con python,93.0545,2.6223
1,2,e2_p,p001.ipynb,competencia_01_ternaria_lags.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,train...",1+2,44,11000,"p001_230047_2.csv,p001_230059_2.csv,p001_23006...","1st_test bajando la cantidad de envios a 11k, ...",93.4285,1.7190
2,3,e3_p,p001.ipynb,competencia_01_ternaria_lags.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,train...",1+2,44,12000,"p001_230047_3.csv,p001_230059_3.csv,p001_23006...","1st_test subiendo la cantidad de envios a 12k,...",93.3130,3.0055
3,4,e4_p,p001.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,train...",1+2,44,11500,"p001_230047_4.csv,p001_230059_4.csv,p001_23006...","cambiamos dataset al q se uso en rds 44, delta...",93.5715,1.5154
4,5,e5_p,p001.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,train...",1+2,44,11000,"p001_230047_5.csv,p001_230059_5.csv,p001_23006...","e4_p pasando de 11.5k envios a 11k envios, python",94.2645,2.1011
5,6,e6_p,p001.ipynb,competencia_01_ternaria_lf_k6_perdida4.csv,"230047,230059,230063,230077,230081",983061,326184,164647,"202103,202104",202108,"ternaria,ternaria_lag1,clase01,fold,azar,train...",1+2,44,10500,"p001_230047_6.csv,p001_230059_6.csv,p001_23006...","e4_p pasando de 11.5k envios a 10.5k envios, p...",95.4525,1.8443
